In [2]:
import os
from PIL import Image

def merge_images_by_keywords(years=None, input_base_path=None, output_base_path=None, spacing=10, year_in_filename=True):
    if not os.path.exists(output_base_path):
        os.makedirs(output_base_path)
    
    period_keywords = ['DJF', 'MAM', 'JJA', 'SON', 'Apr-Sep', 'Annual', 'top-10', 'W126']
    method_keywords = ['model', 'vna_ozone', 'evna_ozone', 'avna_ozone', 'ds_ozone']
    
    if not year_in_filename:
        years = [None]
    
    for year in years:
        period_images = {period: [] for period in period_keywords}
        
        # 组织图片（含空白填充）
        for period in period_keywords:
            max_width, max_height = 0, 0
            method_list = []
            
            # 查找存在的图片并确定最大尺寸
            for method in method_keywords:
                matched_files = []
                for filename in os.listdir(input_base_path):
                    if filename.endswith('.png'):
                        # 检查年份
                        if year is not None and f"{year}_" not in filename:
                            continue
                        # 检查时期和方法
                        if period in filename and method in filename:
                            matched_files.append(filename)
                
                if matched_files:
                    img_path = os.path.join(input_base_path, matched_files[0])
                    try:
                        img = Image.open(img_path)
                        method_list.append((method, img))
                        max_width = max(max_width, img.width)
                        max_height = max(max_height, img.height)
                    except Exception as e:
                        print(f"打开 {img_path} 时出错: {e}")
                        method_list.append((method, None))
                else:
                    method_list.append((method, None))
            
            # 生成空白图片并填充缺失
            for method, img in method_list:
                if img is None:
                    if max_width == 0 or max_height == 0:
                        blank = Image.new('RGB', (100, 100), (255, 255, 255))  # 默认尺寸
                    else:
                        blank = Image.new('RGB', (max_width, max_height), (255, 255, 255))
                    period_images[period].append((method, blank))
                else:
                    period_images[period].append((method, img))
            
            # 按方法顺序排序
            period_images[period].sort(key=lambda x: method_keywords.index(x[0]))
        
        # 合并第一组（季节）
        first_combination = ['DJF', 'MAM', 'JJA', 'SON']
        try:
            all_imgs = [period_images[p] for p in first_combination]
            max_w = max(img.width for p in first_combination for _, img in period_images[p])
            max_h = max(img.height for p in first_combination for _, img in period_images[p])
            
            total_width = max_w * len(method_keywords) + spacing * (len(method_keywords) - 1)
            total_height = max_h * len(first_combination) + spacing * (len(first_combination) - 1)
            
            merged = Image.new('RGB', (total_width, total_height))
            for row_idx, period in enumerate(first_combination):
                for col_idx, (_, img) in enumerate(period_images[period]):
                    x = col_idx * (max_w + spacing)
                    y = row_idx * (max_h + spacing)
                    merged.paste(img, (x, y))
            
            output_name = f"{year}_Seasonal_merged.png" if year else "Seasonal_merged.png"
            merged.save(os.path.join(output_base_path, output_name))
            print(f"成功合并: {output_name}")
        except Exception as e:
            print(f"合并季节组失败: {e}")
        
        # 合并第二组（年度）
        second_combination = ['Apr-Sep', 'Annual', 'top-10', 'W126']
        try:
            all_imgs = [period_images[p] for p in second_combination]
            max_w = max(img.width for p in second_combination for _, img in period_images[p])
            max_h = max(img.height for p in second_combination for _, img in period_images[p])
            
            total_width = max_w * len(method_keywords) + spacing * (len(method_keywords) - 1)
            total_height = max_h * len(second_combination) + spacing * (len(second_combination) - 1)
            
            merged = Image.new('RGB', (total_width, total_height))
            for row_idx, period in enumerate(second_combination):
                for col_idx, (_, img) in enumerate(period_images[period]):
                    x = col_idx * (max_w + spacing)
                    y = row_idx * (max_h + spacing)
                    merged.paste(img, (x, y))
            
            output_name = f"{year}_Annual_merged.png" if year else "Annual_merged.png"
            merged.save(os.path.join(output_base_path, output_name))
            print(f"成功合并: {output_name}")
        except Exception as e:
            print(f"合并年度组失败: {e}")

if __name__ == "__main__":
    years = [2011]
    input_base_path = '/DeepLearning/mnt/shixiansheng/data_fusion/output/boxplots_Alone/'
    output_base_path = '/DeepLearning/mnt/shixiansheng/data_fusion/output/boxplots_Alone_Merged/'
    spacing = 10
    
    merge_images_by_keywords(years, input_base_path, output_base_path, spacing, year_in_filename=True)
    merge_images_by_keywords(None, input_base_path, output_base_path, spacing, year_in_filename=False)

成功合并: 2011_Seasonal_merged.png
成功合并: 2011_Annual_merged.png
成功合并: Seasonal_merged.png
成功合并: Annual_merged.png
